# Phys 235 Schwinger VQS Main Skeleton

Project entry point. Concrete parameters, fixture I/O, workflow ordering, validation display, and report-ready plots live here; algorithm code lives in numbered `.py` modules.


## 1. Dependencies and Local Imports


In [ ]:
# Uncomment in Colab if needed.
# %pip install -q pennylane scipy matplotlib

from pathlib import Path
import importlib
import json
import sys

import numpy as onp
import matplotlib.pyplot as plt

cwd = Path.cwd()
if (cwd / "module1_vqe.py").exists():
    CODE_DIR = cwd
elif (cwd / "code" / "module1_vqe.py").exists():
    CODE_DIR = cwd / "code"
else:
    CODE_DIR = Path("235_Final_Project") / "code"
PROJECT_ROOT = CODE_DIR.parent
TEST_DATA_ROOT = PROJECT_ROOT / "test_data"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import schwinger_core as core_lib
import module1_vqe as module1_lib
import module2_quench as module2_lib
import module3_trotter as module3_lib
import module4_mclachlan_vqs as module4_lib
core_lib = importlib.reload(core_lib)
module1_lib = importlib.reload(module1_lib)
module2_lib = importlib.reload(module2_lib)
module3_lib = importlib.reload(module3_lib)
module4_lib = importlib.reload(module4_lib)

from schwinger_core import compute_observables, exact_time_evolution, state_fidelity
from module1_vqe import Module1Config, module1_acceptance_passed, run_module1_from_config, run_vqe
from module2_quench import Module2Config, module2_acceptance_passed, run_module2_from_config
from module3_trotter import Module3Config, module3_acceptance_passed, run_module3_from_config
from module4_mclachlan_vqs import Module4Config, module4_acceptance_passed, run_module4_from_config


## 2. Project Configuration

All concrete physics and optimizer choices live here. The source paper uses `q_final = 2` for the post-quench external field and `layer_count = 5` for HVA depth.


In [ ]:
PHYSICS_CONFIG = {
    "N": 4,
    "ag": 1.0,
    "m_over_g": 1.0,
    "q_initial": 0.0,
    "q_final": 2.0,
    "g": 1.0,
    "layer_count": 5,
}

VQE_REPRO_CONFIG = {
    "n_restarts": 10,
    "seed": 1234,
    "learning_rate": 0.05,
    "max_steps": 200,
    "grad_tol": 1e-4,
    "stall_window": 100,
    "stall_tol": 1e-9,
    "use_lbfgs_polish": False,
}
VQE_FIXTURE_CONFIG = {**VQE_REPRO_CONFIG, "n_restarts": 5, "max_steps": 50}
ACTIVE_VQE_CONFIG = VQE_REPRO_CONFIG

LAYER_SWEEP_L_VALUES = [1, 2, 3, 4, 5]
LAYER_SWEEP_VQE_CONFIG = {**VQE_FIXTURE_CONFIG, "n_restarts": 20, "max_steps": 50}
RUN_LAYER_SWEEP = False
LAYER_SWEEP_INCLUDE_ZERO_INIT = False
USE_TEST_DATA_INPUT = True
REGENERATE_TEST_DATA = False

MODULE1_FIXTURE_DIR = TEST_DATA_ROOT / "module1_vqe"
MODULE2_FIXTURE_DIR = TEST_DATA_ROOT / "module2_quench"
MODULE3_FIXTURE_DIR = TEST_DATA_ROOT / "module3_trotter"
MODULE4_FIXTURE_DIR = TEST_DATA_ROOT / "module4_mclachlan"
MODULE1_METADATA_PATH = MODULE1_FIXTURE_DIR / "metadata.json"
MODULE2_METADATA_PATH = MODULE2_FIXTURE_DIR / "metadata.json"
MODULE3_METADATA_PATH = MODULE3_FIXTURE_DIR / "metadata.json"
MODULE4_METADATA_PATH = MODULE4_FIXTURE_DIR / "metadata.json"
MODULE1_ARRAYS_PATH = MODULE1_FIXTURE_DIR / "arrays.npz"
MODULE2_ARRAYS_PATH = MODULE2_FIXTURE_DIR / "arrays.npz"
MODULE3_ARRAYS_PATH = MODULE3_FIXTURE_DIR / "arrays.npz"
MODULE4_ARRAYS_PATH = MODULE4_FIXTURE_DIR / "arrays.npz"

TROTTER_CONFIG = {"total_time": 5.0, "n_steps": 100, "n_steps_scan": [10, 20, 40, 80, 160]}
VQS_CONFIG = {
    "total_time": 5.0,
    "n_steps": 400,
    "n_steps_scan": [100, 200, 400, 800],
    "regularization": 1e-8,
    "use_projector": True,
}


## 3. Fixture API


In [ ]:
def load_module_test_data(module_name, test_data_root=TEST_DATA_ROOT):
    module_dir = test_data_root / module_name
    with open(module_dir / "metadata.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)
    arrays = onp.load(module_dir / metadata["arrays_file"], allow_pickle=False)
    return metadata, arrays


def save_metadata_arrays(metadata, arrays_path, **arrays):
    arrays_path.parent.mkdir(parents=True, exist_ok=True)
    with open(arrays_path.parent / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    onp.savez_compressed(arrays_path, **arrays)
    return metadata


def make_module1_config(vqe_config):
    config = Module1Config(
        N=PHYSICS_CONFIG["N"], ag=PHYSICS_CONFIG["ag"], m_over_g=PHYSICS_CONFIG["m_over_g"],
        q_initial=PHYSICS_CONFIG["q_initial"], g=PHYSICS_CONFIG["g"],
        layer_count=PHYSICS_CONFIG["layer_count"], **vqe_config,
    )
    config.validate(); return config


def make_module2_config():
    config = Module2Config(**PHYSICS_CONFIG)
    config.validate(); return config


def make_module3_config():
    config = Module3Config(
        N=PHYSICS_CONFIG["N"], ag=PHYSICS_CONFIG["ag"], m_over_g=PHYSICS_CONFIG["m_over_g"],
        q_final=PHYSICS_CONFIG["q_final"], g=PHYSICS_CONFIG["g"],
        total_time=TROTTER_CONFIG["total_time"], n_steps=TROTTER_CONFIG["n_steps"],
        n_steps_scan=tuple(TROTTER_CONFIG["n_steps_scan"]),
    )
    config.validate(); return config


def make_module4_config():
    config = Module4Config(
        N=PHYSICS_CONFIG["N"], ag=PHYSICS_CONFIG["ag"], m_over_g=PHYSICS_CONFIG["m_over_g"],
        q_final=PHYSICS_CONFIG["q_final"], g=PHYSICS_CONFIG["g"], layer_count=PHYSICS_CONFIG["layer_count"],
        total_time=VQS_CONFIG["total_time"], n_steps=VQS_CONFIG["n_steps"],
        n_steps_scan=tuple(VQS_CONFIG["n_steps_scan"]), regularization=VQS_CONFIG["regularization"],
        use_projector=VQS_CONFIG["use_projector"],
    )
    config.validate(); return config


## 4. Module 1 VQE and Module 2 Quench


In [ ]:
if USE_TEST_DATA_INPUT and MODULE1_METADATA_PATH.exists() and MODULE2_METADATA_PATH.exists() and not REGENERATE_TEST_DATA:
    module1_metadata, module1_arrays = load_module_test_data("module1_vqe")
    module2_metadata, module2_arrays = load_module_test_data("module2_quench")
    module1_config = Module1Config(**module1_metadata["config"])
    module2_config = Module2Config(**module2_metadata["config"])
    print("Loaded Module 1 fixture:", MODULE1_METADATA_PATH)
    print("Loaded Module 2 fixture:", MODULE2_METADATA_PATH)
else:
    module1_config = make_module1_config(ACTIVE_VQE_CONFIG)
    module1_result = run_module1_from_config(module1_config)
    module1_metadata = {
        "schema_version": 2, "module": "module1_vqe",
        "description": "VQE q=0 ground-state fixture for smoke tests.",
        "source_paper": "Nagano, Bapat, Bauer, arXiv:2302.10933",
        "config": module1_result.config.to_dict(), "validation": module1_result.validation,
        "vqe": {"best_energy": module1_result.vqe.best_energy, "adam_energy": module1_result.vqe.adam_energy,
                "exact_ground_energy": module1_result.vqe.exact_ground_energy,
                "exact_max_energy": module1_result.vqe.exact_max_energy,
                "r_E": module1_result.vqe.r_E, "polished": module1_result.vqe.polished},
        "arrays_file": "arrays.npz", "array_keys": ["theta_opt", "statevector", "exact_ground_state", "H_initial_matrix"],
    }
    save_metadata_arrays(module1_metadata, MODULE1_ARRAYS_PATH,
                         theta_opt=module1_result.vqe.theta_opt, statevector=module1_result.vqe.statevector,
                         exact_ground_state=module1_result.vqe.exact_ground_state,
                         H_initial_matrix=module1_result.H_initial_matrix)
    module1_arrays = onp.load(MODULE1_ARRAYS_PATH, allow_pickle=False)

    module2_config = make_module2_config()
    module2_result = run_module2_from_config(module2_config, module1_result.vqe.theta_opt)
    module2_metadata = {
        "schema_version": 2, "module": "module2_quench",
        "description": "q=0 to q=2 quench-ready fixture for smoke tests.",
        "source_paper": "Nagano, Bapat, Bauer, arXiv:2302.10933",
        "config": module2_result.config.to_dict(), "validation": module2_result.validation,
        "quench": {"initial_energy_q0": module2_result.quench.initial_energy_q0,
                   "initial_energy_q2": module2_result.quench.initial_energy_q2,
                   "initial_variance_q2": module2_result.quench.initial_variance_q2,
                   "initial_observables_q2": module2_result.quench.initial_observables_q2},
        "arrays_file": "arrays.npz", "array_keys": ["theta_opt", "psi_0", "H_initial_matrix", "H_final_matrix"],
    }
    save_metadata_arrays(module2_metadata, MODULE2_ARRAYS_PATH,
                         theta_opt=module1_result.vqe.theta_opt, psi_0=module2_result.quench.psi_0,
                         H_initial_matrix=module2_result.quench.H_initial_matrix,
                         H_final_matrix=module2_result.quench.H_final_matrix)
    module2_arrays = onp.load(MODULE2_ARRAYS_PATH, allow_pickle=False)

print("Module 1 r(E):", module1_metadata["validation"]["r_E"])
print("Module 2 q2 variance:", module2_metadata["validation"]["q2_energy_variance"])


### 4.1 VQE Layer-Depth Restart Distribution


In [ ]:
def run_layer_sweep_distribution(l_values, physics_config, vqe_config, include_zero_init=False):
    H_initial = core_lib.build_schwinger_hamiltonian(
        N=physics_config["N"], ag=physics_config["ag"], m_over_g=physics_config["m_over_g"],
        external_field=physics_config["q_initial"], g=physics_config["g"],
    )
    rows = []
    for layer_count in l_values:
        result = run_vqe(H_initial, layer_count, vqe_config["n_restarts"], vqe_config["seed"],
                         vqe_config["learning_rate"], vqe_config["max_steps"], vqe_config["grad_tol"],
                         vqe_config["stall_window"], vqe_config["stall_tol"], False)
        denom = result.exact_max_energy - result.exact_ground_energy
        records = [r for r in result.restart_history if isinstance(r["restart"], int) and (include_zero_init or r["restart"] != 0)]
        restart_r_E = onp.array([(result.exact_max_energy - r["energy"]) / denom for r in records])
        rows.append({"L": layer_count, "r_E_mean": float(onp.mean(restart_r_E)),
                     "r_E_p25": float(onp.percentile(restart_r_E, 25)),
                     "r_E_p75": float(onp.percentile(restart_r_E, 75)),
                     "sample_count": int(restart_r_E.size)})
    return rows

if RUN_LAYER_SWEEP:
    layer_sweep_rows = run_layer_sweep_distribution(LAYER_SWEEP_L_VALUES, PHYSICS_CONFIG, LAYER_SWEEP_VQE_CONFIG, LAYER_SWEEP_INCLUDE_ZERO_INIT)
else:
    layer_sweep_rows = []
    print("Layer sweep skipped. Set RUN_LAYER_SWEEP = True to generate the r(E) distribution plot.")


## 5. Validation Gate and Downstream Input


In [ ]:
module1_ready = module1_acceptance_passed(module1_metadata["validation"])
module2_ready = module2_acceptance_passed(module2_metadata["validation"])
print("Module 1 ready for quench:", module1_ready)
print("Module 2 ready for dynamics:", module2_ready)
assert module1_ready, "Module 1 VQE validation failed."
assert module2_ready, "Module 2 quench validation failed."

module2_downstream_input = {
    "module1_config": module1_metadata["config"],
    "module2_config": module2_metadata["config"],
    "theta_opt": module1_arrays["theta_opt"],
    "psi_0": module2_arrays["psi_0"],
    "H_initial_matrix": module2_arrays["H_initial_matrix"],
    "H_final_matrix": module2_arrays["H_final_matrix"],
}
for key, value in module2_downstream_input.items():
    print(f"{key}:", getattr(value, "shape", value))


## 6. Exact Reference Smoke Output


In [ ]:
times = onp.linspace(0.0, 1.0, 11)
states = exact_time_evolution(module2_downstream_input["H_final_matrix"], module2_downstream_input["psi_0"], times)
exact_reference_observables = [compute_observables(s, N=module2_config.N, ag=module2_config.ag, external_field=module2_config.q_final, g=module2_config.g) for s in states]
print("Generated exact reference states:", states.shape)
print("Initial observables:", exact_reference_observables[0])
print("Final smoke observables:", exact_reference_observables[-1])


## 7. Module 3 Suzuki-Trotter Quench Baseline


In [ ]:
module3_psi_0 = onp.asarray(module2_downstream_input["psi_0"], dtype=complex)
if USE_TEST_DATA_INPUT and MODULE3_METADATA_PATH.exists() and not REGENERATE_TEST_DATA:
    module3_metadata, module3_arrays = load_module_test_data("module3_trotter")
    print("Loaded Module 3 fixture:", MODULE3_METADATA_PATH)
else:
    module3_config = make_module3_config()
    module3_result = run_module3_from_config(module3_config, module3_psi_0)
    module3_metadata = {
        "schema_version": 2, "module": "module3_trotter",
        "description": "Second-order Suzuki-Trotter baseline fixture for the q=2 post-quench dynamics.",
        "source_paper": "Nagano, Bapat, Bauer, arXiv:2302.10933",
        "config": module3_result.config.to_dict(), "validation": module3_result.validation,
        "convergence": {"n_steps_values": list(module3_result.convergence.n_steps_values),
                        "final_fidelity": module3_result.convergence.final_fidelity.tolist(),
                        "final_state_error": module3_result.convergence.final_state_error.tolist(),
                        "order_estimate": module3_result.convergence.order_estimate},
        "arrays_file": "arrays.npz", "array_keys": ["times", "trotter_electric_field", "trotter_chiral_condensate", "trotter_charge", "exact_electric_field", "exact_chiral_condensate", "exact_charge", "fidelity", "conv_n_steps", "conv_final_fidelity", "conv_final_state_error"],
    }
    save_metadata_arrays(module3_metadata, MODULE3_ARRAYS_PATH,
                         times=module3_result.trotter.times, trotter_electric_field=module3_result.trotter.electric_field,
                         trotter_chiral_condensate=module3_result.trotter.chiral_condensate, trotter_charge=module3_result.trotter.charge,
                         exact_electric_field=module3_result.exact.electric_field, exact_chiral_condensate=module3_result.exact.chiral_condensate,
                         exact_charge=module3_result.exact.charge, fidelity=module3_result.fidelity,
                         conv_n_steps=onp.asarray(module3_result.convergence.n_steps_values, dtype=int),
                         conv_final_fidelity=module3_result.convergence.final_fidelity,
                         conv_final_state_error=module3_result.convergence.final_state_error)
    module3_arrays = onp.load(MODULE3_ARRAYS_PATH, allow_pickle=False)
print("Module 3 F(T):", module3_metadata["validation"]["final_fidelity"])
assert module3_acceptance_passed(module3_metadata["validation"])


In [ ]:
m3_times = module3_arrays["times"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(m3_times, module3_arrays["exact_electric_field"], "-", label="exact")
axes[0].plot(m3_times, module3_arrays["trotter_electric_field"], "o", ms=3, label="Trotter")
axes[0].set_xlabel("time"); axes[0].set_ylabel(r"$\langle L(t)angle$"); axes[0].grid(True, alpha=0.3); axes[0].legend()
axes[1].plot(m3_times, module3_arrays["exact_chiral_condensate"], "-", label="exact")
axes[1].plot(m3_times, module3_arrays["trotter_chiral_condensate"], "s", ms=3, label="Trotter")
axes[1].set_xlabel("time"); axes[1].set_ylabel(r"$\langle n(t)angle$"); axes[1].grid(True, alpha=0.3); axes[1].legend()
fig.suptitle(f"Module 3 Trotter vs exact, q={module3_metadata['config']['q_final']}")
fig.tight_layout(); plt.show()

plt.figure(figsize=(6, 4))
plt.plot(m3_times, module3_arrays["fidelity"], "-o", ms=3)
plt.xlabel("time"); plt.ylabel(r"$F(t)$")
plt.title(f"Module 3 Trotter fidelity, n_steps={module3_metadata['config']['n_steps']}")
plt.grid(True, alpha=0.3); plt.ticklabel_format(axis="y", style="plain", useOffset=False); plt.ylim(0.99999, 1.000001); plt.show()


## 8. Module 4 McLachlan Variational Quantum Simulation


In [ ]:
module4_theta0 = onp.asarray(module2_downstream_input["theta_opt"], dtype=float)
if USE_TEST_DATA_INPUT and MODULE4_METADATA_PATH.exists() and not REGENERATE_TEST_DATA:
    module4_metadata, module4_arrays = load_module_test_data("module4_mclachlan")
    print("Loaded Module 4 fixture:", MODULE4_METADATA_PATH)
else:
    module4_config = make_module4_config()
    module4_result = run_module4_from_config(module4_config, module4_theta0)
    module4_metadata = {
        "schema_version": 2, "module": "module4_mclachlan",
        "description": "Projected McLachlan VQS fixture for the q=2 post-quench dynamics.",
        "source_paper": "Nagano, Bapat, Bauer, arXiv:2302.10933",
        "config": module4_result.config.to_dict(), "validation": module4_result.validation,
        "convergence": {"n_steps_values": list(module4_result.convergence.n_steps_values),
                        "final_fidelity": module4_result.convergence.final_fidelity.tolist(),
                        "final_infidelity": module4_result.convergence.final_infidelity.tolist(),
                        "order_estimate": module4_result.convergence.order_estimate},
        "arrays_file": "arrays.npz", "array_keys": ["times", "vqs_electric_field", "vqs_chiral_condensate", "vqs_charge", "exact_electric_field", "exact_chiral_condensate", "exact_charge", "fidelity", "residual", "params", "conv_n_steps", "conv_final_fidelity", "conv_final_infidelity"],
    }
    save_metadata_arrays(module4_metadata, MODULE4_ARRAYS_PATH,
                         times=module4_result.trajectory.times, vqs_electric_field=module4_result.vqs.electric_field,
                         vqs_chiral_condensate=module4_result.vqs.chiral_condensate, vqs_charge=module4_result.vqs.charge,
                         exact_electric_field=module4_result.exact.electric_field, exact_chiral_condensate=module4_result.exact.chiral_condensate,
                         exact_charge=module4_result.exact.charge, fidelity=module4_result.fidelity, residual=module4_result.trajectory.residual,
                         params=module4_result.trajectory.params, conv_n_steps=onp.asarray(module4_result.convergence.n_steps_values, dtype=int),
                         conv_final_fidelity=module4_result.convergence.final_fidelity,
                         conv_final_infidelity=module4_result.convergence.final_infidelity)
    module4_arrays = onp.load(MODULE4_ARRAYS_PATH, allow_pickle=False)
print("Module 4 F(T):", module4_metadata["validation"]["final_fidelity"])
assert module4_acceptance_passed(module4_metadata["validation"])


In [ ]:
m4_times = module4_arrays["times"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(m4_times, module4_arrays["exact_electric_field"], "-", label="exact")
axes[0].plot(module3_arrays["times"], module3_arrays["trotter_electric_field"], "o", ms=3, label="Trotter")
axes[0].plot(m4_times, module4_arrays["vqs_electric_field"], "x", ms=4, label="VQS")
axes[0].set_xlabel("time"); axes[0].set_ylabel(r"$\langle L(t)angle$"); axes[0].grid(True, alpha=0.3); axes[0].legend()
axes[1].plot(m4_times, module4_arrays["exact_chiral_condensate"], "-", label="exact")
axes[1].plot(module3_arrays["times"], module3_arrays["trotter_chiral_condensate"], "s", ms=3, label="Trotter")
axes[1].plot(m4_times, module4_arrays["vqs_chiral_condensate"], "x", ms=4, label="VQS")
axes[1].set_xlabel("time"); axes[1].set_ylabel(r"$\langle n(t)angle$"); axes[1].grid(True, alpha=0.3); axes[1].legend()
fig.suptitle(f"Module 4 VQS vs Trotter vs exact, q={module4_metadata['config']['q_final']}")
fig.tight_layout(); plt.show()

plt.figure(figsize=(6, 4))
plt.plot(m4_times, module4_arrays["fidelity"], "-x", ms=4, label="VQS")
plt.plot(module3_arrays["times"], module3_arrays["fidelity"], "-o", ms=3, label="Trotter")
plt.xlabel("time"); plt.ylabel(r"$F(t)$")
plt.title(f"Fidelity vs exact, VQS n_steps={module4_metadata['config']['n_steps']}")
plt.grid(True, alpha=0.3); plt.ylim(top=1.001); plt.legend(); plt.show()


## 9. Fixture Integrity Smoke Test


In [ ]:
for module_name in ["module1_vqe", "module2_quench", "module3_trotter", "module4_mclachlan"]:
    metadata, arrays = load_module_test_data(module_name)
    assert metadata["schema_version"] == 2
    assert set(metadata["array_keys"]) == set(arrays.files)
    print(module_name, "ok", sorted(arrays.files))
